# Fine-Tune DistilBERT on RavenPack Headlines (TRNA Substitute)

**Goal:** adapt the PhraseBank-trained DistilBERT classifier to RavenPack news headlines, using RavenPack `event_sentiment_score` as the supervision signal (our TRNA substitute).

Companion module: `src/sentiment_ltr/models/ravenpack_sentiment.py`  
Plan reference: `docs/news_sentiment_finetuning_plan.md` (Iteration 4)

---

## Inputs

| Input | Location / detail |
| --- | --- |
| **RavenPack article export** | `data/raw/news/ravenpack/{ticker}_articles_2003_2014.parquet` — built by `notebooks/fetch_news_articles.ipynb` |
| **Required columns** | `headline`, `event_sentiment_score`, `article_date` (or `article_time`) |
| **Starting checkpoint** (recommended) | `data/models/phrasebank_distilbert_best/` — DistilBERT fine-tuned on Financial PhraseBank |
| **Fallback init** | `distilbert-base-uncased` if no PhraseBank checkpoint exists |
| **Label mapping** | `event_sentiment_score` → `negative` / `neutral` / `positive` (threshold ±0.05) |
| **Train / val / test split** | Time-based: train ≤ 2011, validation = 2012, test ≥ 2013 |
| **Hyperparameters** | `lr=2e-5`, `batch_size=16`, `max_length=128`, `epochs=2` (default) |

**Prerequisites:** conda env `sentiment-ltr-paper` with `requirements-finetuning.txt` installed; at least one ticker with a rich RavenPack export (currently **AAPL**: ~69k labeled headlines).

---

## Steps

1. **Environment check** — confirm `torch`, `transformers`, `datasets`, and GPU/MPS device.
2. **Discover data** — list `*_articles_*.parquet` under `data/raw/news/ravenpack/`; pick ticker(s).
3. **Load & inspect** — labeled rows (headline + score), class balance, split row counts.
4. **Build HF dataset** — map headlines to `sentence`, scores to 3-class `label`; time-based splits.
5. **Load init checkpoint** — PhraseBank weights (recommended) or `distilbert-base-uncased`.
6. **Tokenize** — `max_length=128`, fixed padding (same as PhraseBank pipeline).
7. **Train** — Hugging Face `Trainer`, 2 epochs, eval each epoch, `load_best_model_at_end` on **validation macro-F1**.
8. **Evaluate** — report validation + **test** accuracy and macro-F1.
9. **Save** — write checkpoint + `metrics.json` for the app / batch scoring.

---

## Expected outputs

| Output | Path / description |
| --- | --- |
| **Fine-tuned model** | `data/models/ravenpack_distilbert_best/` (`config.json`, tokenizer, weights) |
| **Training metrics** | `data/models/ravenpack_distilbert_best/metrics.json` — train loss, val/test `eval_f1`, `eval_accuracy`, hyperparams, tickers used |
| **Console summary** | Split sizes (e.g. AAPL: ~38k train / ~13k val / ~18k test), class balance, final test macro-F1 |
| **Downstream use** | Sentiment Lab → *Fine-tune on RavenPack headlines*; future batch-scoring of cached headlines (Iteration 4.2) |

**Success criteria:** training completes without error; test macro-F1 is reported; checkpoint loads for inference on new headlines.


In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

%load_ext autoreload
%autoreload 2

from sentiment_ltr.models.phrasebank_sentiment import (
    DEFAULT_MODEL_DIR,
    MAX_LENGTH,
    MODEL_NAME,
    label_maps,
    load_classifier,
    load_metrics,
    load_phrasebank,
    model_is_saved,
    phrasebank_data_provenance,
    phrasebank_probability_chart_frame,
    phrasebank_sample_inference_frame,
    phrasebank_tokenization_example_frame,
    phrasebank_tokenizer_settings_frame,
    predict_sentences,
)
from sentiment_ltr.models.ravenpack_sentiment import (
    DEFAULT_RAVENPACK_MODEL_DIR,
    ID2LABEL as RP_ID2LABEL,
    RAVENPACK_NEWS_DIR,
    SENTIMENT_SCORE_THRESHOLD,
    SPLIT_SOURCE,
    assign_time_split,
    discover_ravenpack_article_files,
    load_ravenpack_labeled_frame,
    ravenpack_class_balance,
    ravenpack_data_provenance,
    ravenpack_model_is_saved,
    ravenpack_split_summary,
    ravenpack_to_hf_dataset,
    validate_phrasebank_ravenpack_label_alignment,
)
from sentiment_ltr.provenance import (
    build_checkpoint_provenance,
    get_model_config_info,
    get_tokenizer_provenance,
    print_provenance_summary,
    save_provenance,
)
from sentiment_ltr.viz import split_series_distribution_figures
from sentiment_ltr.wandb_logging import log_ravenpack_diagnostics

In [2]:
# ── 1. Load PhraseBank checkpoint & display tokenizer / model settings ───────
if not model_is_saved(DEFAULT_MODEL_DIR):
    raise FileNotFoundError(
        f"No checkpoint at {DEFAULT_MODEL_DIR}. Run notebooks/liquidAI_prep.ipynb first."
    )

tokenizer, model, device = load_classifier(DEFAULT_MODEL_DIR)
metrics = load_metrics(DEFAULT_MODEL_DIR)

phrasebank_tokenizer_settings_frame(tokenizer, model, metrics)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,setting,value
0,base_model (HF hub id),distilbert-base-uncased
1,tokenizer_class,BertTokenizer
2,tokenizer_loaded_from,/Users/armandoordoricadelatorre/Documents/U of...
3,model_type,distilbert
4,vocab_size,30522
5,padding_side,right
6,tokenizer.model_max_length,512
7,train/infer max_length,128
8,training truncation,True
9,training padding,max_length


In [3]:
# ── 2. Sample inferences on PhraseBank validation & test splits ──────────────
N = 2  # samples per split

sample_results = phrasebank_sample_inference_frame(tokenizer, model, device, n=N)
display(sample_results)

# ── 3. Training-style tokenization for the first sample sentence ─────────────
train_max_length = metrics.get("max_length", MAX_LENGTH)
phrasebank_tokenization_example_frame(tokenizer, sample_results["sentence"].iloc[0], train_max_length)


,split,sentence,label,pred,p(negative),p(neutral),p(positive),match
0,validation,BasWare Order Matching automatically matches p...,neutral,neutral,0.0055,0.9822,0.0123,True
1,validation,"Elisa Corporation , headquartered in Helsinki ...",neutral,neutral,0.0043,0.9845,0.0112,True
2,test,Net sales are expected to remain on the same l...,neutral,neutral,0.0049,0.9841,0.0110,True
3,test,The executive said that countries such as Braz...,neutral,neutral,0.0041,0.9299,0.0661,True


,token,input_id,attention_mask
0,[CLS],101,1
1,bas,19021,1
2,##ware,8059,1
3,order,2344,1
4,matching,9844,1
...,...,...,...
123,[PAD],0,0
124,[PAD],0,0
125,[PAD],0,0
126,[PAD],0,0


In [4]:
# ── 4. Class probability distribution across PhraseBank splits ───────────────
CLASS_ORDER = ["negative", "neutral", "positive"]

long_probs = phrasebank_probability_chart_frame()
split_order = long_probs["split"].cat.categories.tolist()

fig_box, fig_p50, p50 = split_series_distribution_figures(
    long_probs,
    x="split",
    y="probability",
    series="class",
    category_orders={"split": split_order, "class": CLASS_ORDER},
    axis_labels={
        "split": "PhraseBank split",
        "probability": "Predicted probability",
        "p50": "Median predicted probability",
        "class": "Class",
    },
    box_title="Class probabilities by split (box & whisker)",
    median_title="Median class probability by split (p50)",
    median_col="p50",
    x_hover_label="PhraseBank split",
    series_hover_label="Class",
    y_hover_label="Predicted probability",
    median_y_hover_label="Median predicted probability",
)
fig_box.show()
fig_p50.show()

# ── 5. Verify PhraseBank checkpoint, dataset, and RavenPack labels all match ─
raw = load_phrasebank()
_, dataset_id2label, _ = label_maps(raw)

validate_phrasebank_ravenpack_label_alignment()


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

PhraseBank checkpoint config (model.config)
       label
id          
0   negative
1    neutral
2   positive

PhraseBank HF dataset ClassLabel (should match checkpoint)
       label
id          
0   negative
1    neutral
2   positive

RavenPack fine-tune target labels (same 3-class schema)
       label
id          
0   negative
1    neutral
2   positive

✓ All three id2label maps match — RavenPack labels align with PhraseBank head.


{'checkpoint': {0: 'negative', 1: 'neutral', 2: 'positive'},
 'dataset': {0: 'negative', 1: 'neutral', 2: 'positive'},
 'ravenpack': {0: 'negative', 1: 'neutral', 2: 'positive'}}

## RavenPack inputs & outputs (AAPL)

Loads the rich local export from `notebooks/fetch_news_articles.ipynb`:

| Stage | What | Detail |
| --- | --- | --- |
| **Input file** | `data/raw/news/ravenpack/aapl_articles_2003_2014.parquet` | ~311k article rows × 23 columns (headline, scores, prices, metadata) |
| **Required columns** | `headline`, `event_sentiment_score`, `article_date` | Used by `load_ravenpack_labeled_frame()` |
| **Label rule** | `event_sentiment_score` → 3 classes | `> +0.05` positive, `< -0.05` negative, else neutral |
| **Labeled frame** | Deduped rows with `label_name` + `label` | ~69k usable headlines for fine-tuning |
| **HF output** | `DatasetDict` with `sentence` + `label` | Time split: train ≤2011, val 2012, test ≥2013 |

Module: `src/sentiment_ltr/models/ravenpack_sentiment.py`


In [5]:
TICKER = "AAPL"

# ── Input: local parquet export ──────────────────────────────────────────────
article_paths = discover_ravenpack_article_files([TICKER])
if not article_paths:
    raise FileNotFoundError(
        f"No RavenPack export for {TICKER} under {RAVENPACK_NEWS_DIR}. "
        "Run notebooks/fetch_news_articles.ipynb first."
    )
article_path = article_paths[0]

print("INPUT")
print(f"  file:        {article_path.relative_to(PROJECT_ROOT)}")
print(f"  label rule:  |event_sentiment_score| > {SENTIMENT_SCORE_THRESHOLD} → pos/neg; else neutral")
print(f"  splits:      {SPLIT_SOURCE}\n")

raw_rp = pd.read_parquet(article_path)
display(
    pd.Series(
        {
            "rows": f"{len(raw_rp):,}",
            "columns": raw_rp.shape[1],
            "date_min": str(raw_rp["article_date"].min()),
            "date_max": str(raw_rp["article_date"].max()),
        },
        name="raw export summary",
    ).to_frame("value")
)
print("Raw column dtypes:")
display(raw_rp.dtypes.to_frame("dtype"))
print("Sample raw rows (training-relevant columns):")
display(
    raw_rp[
        ["ticker", "article_date", "headline", "event_sentiment_score", "relevance_score", "story_id"]
    ].head(5)
)

# ── Labeled frame (module output) ────────────────────────────────────────────
labeled = load_ravenpack_labeled_frame([TICKER])

print("\nLABELED FRAME (after score→label, headline filter, dedupe)")
display(
    pd.Series(
        {
            "rows": f"{len(labeled):,}",
            "dropped_from_raw": f"{len(raw_rp) - len(labeled):,}",
            "columns": labeled.shape[1],
        },
        name="labeled summary",
    ).to_frame("value")
)
display(ravenpack_class_balance(labeled))
display(ravenpack_split_summary(labeled))
print("Sample labeled rows:")
display(
    labeled[
        ["ticker", "article_date", "headline", "event_sentiment_score", "label_name", "label"]
    ].head(5)
)

# ── Hugging Face DatasetDict (fine-tuning input) ──────────────────────────────
hf_ds = ravenpack_to_hf_dataset(labeled)

print("\nHF DATASET OUTPUT (what train_ravenpack() consumes)")
hf_summary = pd.DataFrame(
    {
        "split": list(hf_ds.keys()),
        "rows": [len(hf_ds[s]) for s in hf_ds],
        "columns": [", ".join(hf_ds[s].column_names) for s in hf_ds],
    }
)
display(hf_summary)
print("Sample train rows (`sentence` = headline, `label` = 0/1/2):")
display(hf_ds["train"].to_pandas().head(5))


INPUT
  file:        data/raw/news/ravenpack/aapl_articles_2003_2014.parquet
  label rule:  |event_sentiment_score| > 0.05 → pos/neg; else neutral
  splits:      RavenPack time split: train ≤2011, val 2012, test ≥2013 (|score|>0.05 → label; else neutral)



,value
rows,"311,041"
columns,23
date_min,2003-01-02 00:00:00
date_max,2014-12-31 00:00:00


Raw column dtypes:


,dtype
ticker,object
source,object
article_time,"datetime64[ns, UTC]"
article_date,datetime64[ns]
week_start,datetime64[ns]
headline,object
event_text,object
story_id,object
source_code,object
relevance_score,float64


Sample raw rows (training-relevant columns):


...

Casting the dataset:   0%|          | 0/18154 [00:00<?, ? examples/s]


HF DATASET OUTPUT (what train_ravenpack() consumes)


,split,rows,columns
0,train,37832,"sentence, label"
1,validation,12736,"sentence, label"
2,test,18154,"sentence, label"


Sample train rows (`sentence` = headline, `label` = 0/1/2):


,sentence,label
0,PRESS RELEASE: Apple Plans Macworld Expo Prese...,1
1,PRESS RELEASE: Apple To Open Store In Pasadena...,2
2,DJ Apple's Jobs: Apple Stores FY1Q Revenue $14...,1
3,DJ News Highlights: Apple Stores FY1Q Revenue ...,1
4,PRESS RELEASE: Microsoft Unveils Pdts For Mac ...,2


## Fine-tune DistilBERT on RavenPack headlines

Runs `train_ravenpack()` from `ravenpack_sentiment.py` — a single call that:

1. Re-uses the labeled frame and `hf_ds` built above
2. Initialises from the **PhraseBank checkpoint** (warm start — model already knows financial language)
3. Runs HF `Trainer` for 2 epochs, selecting the best checkpoint on **validation macro-F1** each epoch
4. Saves weights + tokenizer to `data/models/ravenpack_distilbert_best/`
5. Writes `metrics.json` with train loss, val/test macro-F1, hyperparams, and tickers used

**Baseline to beat:** PhraseBank out-of-box on RavenPack test → **27.5% macro-F1** (54.6 pp domain-shift gap)

**Hyperparameters (defaults — edit the config cell below to override):**

| Param | Value | Rationale |
| --- | --- | --- |
| `init_from_phrasebank` | `True` | Warm start — preserves financial-language representations |
| `num_train_epochs` | `2` | Sufficient for a strong first run; 3+ risks overfit on ~38k rows |
| `learning_rate` | `2e-5` | Conservative for continued fine-tune from an existing checkpoint |
| `per_device_train_batch_size` | `16` | Matches PhraseBank training setup |
| `seed` | `42` | Reproducibility |

**After training completes:** go to the **AAPL inference** section below, set `FORCE_CHECKPOINT = "ravenpack"`, and re-run all cells from there down to compare results.


In [17]:
# ── A. Environment check ─────────────────────────────────────────────────────
import torch
import transformers
import datasets as hf_datasets

from sentiment_ltr.models.phrasebank_sentiment import device_report, finetuning_deps_available

if not finetuning_deps_available():
    raise EnvironmentError(
        "torch / transformers / datasets are not importable. "
        "Activate the `sentiment-ltr-paper` conda env and install "
        "`requirements-finetuning.txt` before running this section."
    )

dr = device_report()

env_summary = pd.Series({
    "torch":             torch.__version__,
    "transformers":      transformers.__version__,
    "datasets":          hf_datasets.__version__,
    "device":            dr["selected"],
    "device_name":       dr["device_name"],
    "cuda_available":    dr["cuda_available"],
    "mps_available":     dr["mps_available"],
    "init_checkpoint":   str(DEFAULT_MODEL_DIR.relative_to(PROJECT_ROOT))
                         if model_is_saved(DEFAULT_MODEL_DIR)
                         else f"⚠ PhraseBank checkpoint missing — will init from {MODEL_NAME}",
    "train_rows (AAPL)": f"~{len(hf_ds['train']):,}",
    "val_rows (AAPL)":   f"~{len(hf_ds['validation']):,}",
    "test_rows (AAPL)":  f"~{len(hf_ds['test']):,}",
}, name="value")

display(env_summary.to_frame())
print("\n✓ Environment ready for fine-tuning.")


,value
torch,2.12.1
transformers,5.12.1
datasets,5.0.0
device,mps
device_name,Apple GPU (Metal / MPS)
cuda_available,False
mps_available,True
init_checkpoint,data/models/phrasebank_distilbert_best
train_rows (AAPL),"~37,832"
val_rows (AAPL),"~12,736"



✓ Environment ready for fine-tuning.


In [18]:
# ── B. Training config (edit here to override defaults) ──────────────────────
from sentiment_ltr.models.ravenpack_sentiment import train_ravenpack

TRAIN_TICKERS          = [TICKER]   # extend to e.g. ["AAPL", "MSFT"] for multi-ticker
TRAIN_EPOCHS           = 2          # 2 = safe first run; 3 risks overfit on ~38k rows
TRAIN_LR               = 2e-5       # conservative for warm-start from PhraseBank
TRAIN_BATCH_SIZE       = 16         # matches PhraseBank setup
TRAIN_SEED             = 42
TRAIN_FROM_PHRASEBANK  = True       # False → init from distilbert-base-uncased

# ── B. Run fine-tuning ────────────────────────────────────────────────────────
print(f"Starting fine-tune on {TRAIN_TICKERS} …")
print(f"  init:    {'PhraseBank checkpoint' if TRAIN_FROM_PHRASEBANK else MODEL_NAME}")
print(f"  epochs:  {TRAIN_EPOCHS}   lr: {TRAIN_LR:g}   batch: {TRAIN_BATCH_SIZE}   seed: {TRAIN_SEED}")
print(f"  output:  {DEFAULT_RAVENPACK_MODEL_DIR.relative_to(PROJECT_ROOT)}\n")

rp_train_metrics = train_ravenpack(
    tickers=TRAIN_TICKERS,
    init_from_phrasebank=TRAIN_FROM_PHRASEBANK,
    num_train_epochs=TRAIN_EPOCHS,
    learning_rate=TRAIN_LR,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    seed=TRAIN_SEED,
)

print("\n✓ Training complete.")


Starting fine-tune on ['AAPL'] …
  init:    PhraseBank checkpoint
  epochs:  2   lr: 2e-05   batch: 16   seed: 42
  output:  data/models/ravenpack_distilbert_best



Casting the dataset:   0%|          | 0/37832 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/12736 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/18154 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.584653,0.877032,2,0.638427,0.497669


/Users/armandoordoricadelatorre/miniconda/envs/sentiment-ltr-paper/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.584653,0.973096,2,0.623719,0.491119


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Training complete.


In [19]:
# ── C. Post-training metrics summary ─────────────────────────────────────────
_val  = rp_train_metrics.get("validation", {})
_test = rp_train_metrics.get("test", {})

summary = pd.Series({
    "tickers":               ", ".join(rp_train_metrics.get("tickers", [])),
    "labeled_rows":          f"{rp_train_metrics['labeled_rows']:,}",
    "init_checkpoint":       rp_train_metrics["init_checkpoint"],
    "epochs":                rp_train_metrics["epochs"],
    "learning_rate":         rp_train_metrics["learning_rate"],
    "batch_size":            rp_train_metrics["per_device_train_batch_size"],
    "train_loss":            f"{rp_train_metrics['train_loss']:.4f}",
    "train_runtime_min":     f"{rp_train_metrics['train_runtime_s'] / 60:.1f}",
    "val  macro-F1":         f"{_val.get('eval_f1',  float('nan')):.1%}",
    "val  accuracy":         f"{_val.get('eval_accuracy', float('nan')):.1%}",
    "test macro-F1  ← key":  f"{_test.get('eval_f1', float('nan')):.1%}",
    "test accuracy":         f"{_test.get('eval_accuracy', float('nan')):.1%}",
    "device":                rp_train_metrics.get("device", ""),
    "saved_at":              rp_train_metrics.get("saved_at", ""),
    "output_dir":            str(DEFAULT_RAVENPACK_MODEL_DIR.relative_to(PROJECT_ROOT)),
}, name="value")

display(summary.to_frame())

_baseline = 0.275   # PhraseBank out-of-box on RavenPack test (cell 9)
_new_f1   = _test.get("eval_f1", float("nan"))
_gain     = _new_f1 - _baseline

print(f"\nBaseline (PhraseBank OOB on RavenPack test) : {_baseline:.1%}")
print(f"Fine-tuned (RavenPack test macro-F1)        : {_new_f1:.1%}")
print(f"Improvement                                 : {_gain:+.1%} pp")
print(
    "\n→ Next step: set FORCE_CHECKPOINT = 'ravenpack' in the inference cell below "
    "and re-run all cells from there down to update all charts and W&B."
)


,value
tickers,AAPL
labeled_rows,"68,722"
init_checkpoint,/Users/armandoordoricadelatorre/Documents/U of...
epochs,2
learning_rate,0.00002
batch_size,16
train_loss,0.6949
train_runtime_min,16.6
val macro-F1,49.8%
val accuracy,63.8%



Baseline (PhraseBank OOB on RavenPack test) : 27.5%
Fine-tuned (RavenPack test macro-F1)        : 49.1%
Improvement                                 : +21.6% pp

→ Next step: set FORCE_CHECKPOINT = 'ravenpack' in the inference cell below and re-run all cells from there down to update all charts and W&B.


## AAPL inference: predicted vs RavenPack actual labels

Score headlines with the best available checkpoint (RavenPack if trained, else PhraseBank) and compare `pred` to RavenPack `label_name` (`event_sentiment_score` → negative / neutral / positive).

In [41]:
# ── Config ───────────────────────────────────────────────────────────────────
# FORCE_CHECKPOINT = "phrasebank" → PhraseBank baseline applied to RavenPack headlines
# FORCE_CHECKPOINT = "ravenpack"  → requires a trained RavenPack checkpoint
# FORCE_CHECKPOINT = "auto"       → prefers RavenPack if saved, else PhraseBank
FORCE_CHECKPOINT = "ravenpack"
EVAL_SPLIT = "test"   # "train" | "validation" | "test" | None (= all)
BATCH_SIZE = 64
MAX_ROWS = None        # set to e.g. 2000 for a quick smoke test

if FORCE_CHECKPOINT == "phrasebank":
    model_dir = DEFAULT_MODEL_DIR
    ckpt_label = "PhraseBank (out-of-box on RavenPack)"
elif FORCE_CHECKPOINT == "ravenpack":
    if not ravenpack_model_is_saved():
        raise FileNotFoundError("No RavenPack checkpoint — train first or set FORCE_CHECKPOINT='phrasebank'")
    model_dir = DEFAULT_RAVENPACK_MODEL_DIR
    ckpt_label = "RavenPack fine-tuned"
else:  # auto
    use_ravenpack = ravenpack_model_is_saved()
    model_dir = DEFAULT_RAVENPACK_MODEL_DIR if use_ravenpack else DEFAULT_MODEL_DIR
    ckpt_label = "RavenPack fine-tuned" if use_ravenpack else "PhraseBank (out-of-box)"

print(f"Checkpoint : {ckpt_label}")
print(f"  path     : {model_dir.relative_to(PROJECT_ROOT)}")

tokenizer, model, device = load_classifier(model_dir)

eval_df = labeled.copy()
eval_df["split"] = assign_time_split(eval_df["article_date"])
if EVAL_SPLIT:
    eval_df = eval_df[eval_df["split"] == EVAL_SPLIT].reset_index(drop=True)
if MAX_ROWS and len(eval_df) > MAX_ROWS:
    eval_df = eval_df.sample(MAX_ROWS, random_state=42).reset_index(drop=True)

print(f"\nScoring {len(eval_df):,} {TICKER} headlines" + (f" ({EVAL_SPLIT} split)" if EVAL_SPLIT else ""))

headlines = eval_df["headline"].tolist()
pred_chunks: list[pd.DataFrame] = []
for start in range(0, len(headlines), BATCH_SIZE):
    pred_chunks.append(predict_sentences(headlines[start : start + BATCH_SIZE], tokenizer, model, device))
preds = pd.concat(pred_chunks, ignore_index=True)

results = (
    eval_df[["split", "article_date", "headline", "event_sentiment_score", "label_name"]]
    .join(preds.drop(columns=["sentence"]))
    .rename(columns={"label_name": "actual"})
    .assign(match=lambda df: df["actual"] == df["pred"])
)

label_order = [RP_ID2LABEL[i] for i in sorted(RP_ID2LABEL)]
y_true = results["actual"]
y_pred = results["pred"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0)

print("\n=== Performance metrics ===")
print(f"  accuracy:  {acc:.1%}")
print(f"  macro-F1:  {macro_f1:.1%}")
print("\nClassification report (actual = RavenPack score labels):")
print(classification_report(y_true, y_pred, labels=label_order, digits=3, zero_division=0))

cm = pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=label_order),
    index=pd.Index(label_order, name="actual"),
    columns=pd.Index(label_order, name="pred"),
)
cm_pct = cm.div(cm.sum(axis=1), axis=0).mul(100)

display(
    cm.style.format("{:,.0f}")
    .bar(align="mid", color=["red", "lightgreen"])
    .set_caption(f"Confusion matrix — counts ({ckpt_label}, {TICKER} {EVAL_SPLIT})")
)
display(
    cm_pct.round(1)
    .style.format("{:.1f}%")
    .bar(align="mid", color=["red", "lightgreen"], vmin=0, vmax=100)
    .set_caption("Confusion matrix — % of actual class (row-normalized)")
)

# ── Colour-coded mismatch sample ─────────────────────────────────────────────
_SENTIMENT_BG = {
    "negative": "background-color: #fecaca; color: #7f1d1d",
    "neutral":  "background-color: #e2e8f0; color: #1e293b",
    "positive": "background-color: #bbf7d0; color: #14532d",
}

def _color_sentiment(val):
    return _SENTIMENT_BG.get(str(val), "")

mismatches = results.loc[~results["match"]]
_prob_cols = ["p(negative)", "p(neutral)", "p(positive)"]
_label_cols = ["actual", "pred"]
_mismatch_sample = mismatches.sample(min(10, len(mismatches)), random_state=42)[
    ["article_date", "headline", "event_sentiment_score", "actual", "pred"] + _prob_cols
]
print("Random mismatches (🟥 negative · ⬜ neutral · 🟩 positive):")
display(
    _mismatch_sample.style
    .applymap(_color_sentiment, subset=_label_cols)
    .format("{:.3f}", subset=_prob_cols)
    .bar(subset=_prob_cols, align="mid", color=["red", "lightgreen"])
)

Checkpoint : RavenPack fine-tuned
  path     : data/models/ravenpack_distilbert_best


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Scoring 18,154 AAPL headlines (test split)

=== Performance metrics ===
  accuracy:  62.4%
  macro-F1:  49.1%

Classification report (actual = RavenPack score labels):
              precision    recall  f1-score   support

    negative      0.582     0.355     0.441      4926
     neutral      0.443     0.219     0.293      2730
    positive      0.651     0.855     0.739     10498

    accuracy                          0.624     18154
   macro avg      0.558     0.476     0.491     18154
weighted avg      0.601     0.624     0.591     18154


=== Performance metrics ===
  accuracy:  62.4%
  macro-F1:  49.1%

Classification report (actual = RavenPack score labels):
              precision    recall  f1-score   support

    negative      0.582     0.355     0.441      4926
     neutral      0.443     0.219     0.293      2730
    positive      0.651     0.855     0.739     10498

    accuracy                          0.624     18154
   macro avg      0.558     0.476     0.491     18154

pred,negative,neutral,positive
actual,,,
negative,"1,749",200,"2,977"
neutral,286,599,"1,845"
positive,969,554,"8,975"


pred,negative,neutral,positive
actual,,,
negative,35.5%,4.1%,60.4%
neutral,10.5%,21.9%,67.6%
positive,9.2%,5.3%,85.5%


Random mismatches:


,article_date,headline,event_sentiment_score,actual,pred,p(negative),p(neutral),p(positive)
11682,2014-02-10,Australian Morning Briefing: U.S. Stocks Claw ...,0.59,positive,negative,0.5037,0.0486,0.4478
4469,2013-04-30,2nd UPDATE: Apple Prices Record $17 Billion De...,0.00,neutral,positive,0.0160,0.0525,0.9314
12398,2014-04-10,Nymex Crude Biased Down Near Term; $103.10 Sup...,0.00,neutral,positive,0.3220,0.1611,0.5170
8571,2013-10-01,CHART Apple Intraday: the downside prevails.,-0.43,negative,positive,0.3565,0.1298,0.5137
16259,2014-09-25,MARKET SNAPSHOT: U.S. Stocks Fall; On Track Fo...,-0.51,negative,positive,0.3369,0.0266,0.6365
8606,2013-10-01,"Smartphones: Trade-Ins, BlackBerry Demise to D...",-0.56,negative,positive,0.2445,0.1120,0.6435
7978,2013-09-11,"U.S. Stocks Broadly Higher; Tech, Utilities So...",-0.11,negative,positive,0.2804,0.0268,0.6928
7892,2013-09-11,IPhone 5C Doesn't Stand for Cheap - Market Talk,-0.11,negative,positive,0.1429,0.1810,0.6761
14572,2014-07-25,"Goldman Sachs Cuts Views on Stocks, Credit -- ...",0.62,positive,negative,0.6992,0.0529,0.2479
9907,2013-11-11,Apple Supplier Pegatron Posts 53% Rise in 3Q N...,0.00,neutral,positive,0.0108,0.0193,0.9699


## Label distribution: PhraseBank (train) vs RavenPack (inference)

Compares class prevalence across three views:
1. **PhraseBank** — the domain the model was trained on (per split + total)
2. **RavenPack actual** — true labels derived from `event_sentiment_score` thresholding (out-of-domain)
3. **RavenPack predicted** — what the model actually outputs on the eval split

Large gaps between PhraseBank and RavenPack actual reveal label-distribution shift.
Large gaps between actual and predicted reveal model bias toward the training distribution.

In [42]:
import plotly.graph_objects as go

_CLASS_ORDER = ["negative", "neutral", "positive"]

# ── 1. PhraseBank distribution per split (training domain) ───────────────────
pb_rows = []
for split_name in ["train", "validation", "test"]:
    ds_split = raw[split_name].to_pandas()
    total    = len(ds_split)
    counts   = ds_split["label"].map(dataset_id2label).value_counts()
    for cls in _CLASS_ORDER:
        n = int(counts.get(cls, 0))
        pb_rows.append({"split": split_name, "label": cls, "count": n, "pct": 100 * n / total})
pb_dist = pd.DataFrame(pb_rows)

pb_total = (
    pb_dist.groupby("label", as_index=False)[["count"]].sum()
    .assign(pct=lambda d: d["count"] / d["count"].sum() * 100, split="ALL")
)

# ── 2. RavenPack actual distribution (all splits — full labeled frame) ────────
rp_actual_vc = labeled["label_name"].value_counts().reindex(_CLASS_ORDER, fill_value=0)
rp_actual = pd.DataFrame({
    "label": _CLASS_ORDER,
    "count": rp_actual_vc.values.tolist(),
}).assign(pct=lambda d: d["count"] / d["count"].sum() * 100)

# ── 3. RavenPack predicted distribution (eval split from previous cell) ───────
rp_pred_vc = results["pred"].value_counts().reindex(_CLASS_ORDER, fill_value=0)
rp_pred = pd.DataFrame({
    "label": _CLASS_ORDER,
    "count": rp_pred_vc.values.tolist(),
}).assign(pct=lambda d: d["count"] / d["count"].sum() * 100)

# ── Summary table ─────────────────────────────────────────────────────────────
print("=== PhraseBank label distribution by split ===")
display(
    pb_dist.pivot_table(index="label", columns="split", values=["count", "pct"], aggfunc="sum")
    .reindex(_CLASS_ORDER)
    .round(1)
)

cmp = pd.DataFrame({
    "PhraseBank total":              pb_total.set_index("label")["count"].reindex(_CLASS_ORDER),
    "PhraseBank total %":            pb_total.set_index("label")["pct"].reindex(_CLASS_ORDER).round(1),
    "RavenPack actual count":        rp_actual.set_index("label")["count"].reindex(_CLASS_ORDER),
    "RavenPack actual %":            rp_actual.set_index("label")["pct"].reindex(_CLASS_ORDER).round(1),
    f"RavenPack predicted count":    rp_pred.set_index("label")["count"].reindex(_CLASS_ORDER),
    f"RavenPack predicted %":        rp_pred.set_index("label")["pct"].reindex(_CLASS_ORDER).round(1),
})
print(f"\n=== Totals comparison (RavenPack eval split: {EVAL_SPLIT}, checkpoint: {ckpt_label}) ===")
display(cmp)

# ── Bar chart: three-way comparison ──────────────────────────────────────────
fig = go.Figure()
traces = [
    ("PhraseBank (all splits)", pb_total.set_index("label").reindex(_CLASS_ORDER)),
    ("RavenPack actual (all splits)", rp_actual.set_index("label").reindex(_CLASS_ORDER)),
    (f"RavenPack predicted ({ckpt_label}, {EVAL_SPLIT})", rp_pred.set_index("label").reindex(_CLASS_ORDER)),
]
for name, df in traces:
    fig.add_trace(go.Bar(
        name=name,
        x=_CLASS_ORDER,
        y=df["pct"].tolist(),
        text=[f"{p:.1f}%<br>n={n:,}" for p, n in zip(df["pct"], df["count"])],
        textposition="outside",
    ))

fig.update_layout(
    barmode="group",
    title="Label distribution shift: PhraseBank (train) → RavenPack (out-of-domain)",
    xaxis_title="Sentiment class",
    yaxis=dict(title="% of dataset", range=[0, 105]),
    legend=dict(
        orientation="v",
        yanchor="middle", y=0.5,
        xanchor="left",   x=1.02,
    ),
    height=500,
    margin=dict(t=60, r=280),
)
fig.show()

# ── Bar chart: PhraseBank per-split breakdown ─────────────────────────────────
fig2 = go.Figure()
for split_name in ["train", "validation", "test"]:
    sdf = pb_dist[pb_dist["split"] == split_name].set_index("label").reindex(_CLASS_ORDER)
    fig2.add_trace(go.Bar(
        name=f"PhraseBank {split_name}",
        x=_CLASS_ORDER,
        y=sdf["pct"].tolist(),
        text=[f"{p:.1f}%<br>n={n:,}" for p, n in zip(sdf["pct"], sdf["count"])],
        textposition="outside",
    ))
fig2.update_layout(
    barmode="group",
    title="PhraseBank label distribution by split",
    xaxis_title="Sentiment class",
    yaxis=dict(title="% of split", range=[0, 105]),
    height=430,
)
fig2.show()

=== PhraseBank label distribution by split ===


count                    pct                 
split     test train validation  test train validation
label                                                 
negative   125   382         97  12.9  12.3       12.5
neutral    565  1852        462  58.2  59.7       59.5
positive   280   866        217  28.9  27.9       28.0


=== Totals comparison (RavenPack eval split: test, checkpoint: RavenPack fine-tuned) ===


,PhraseBank total,PhraseBank total %,RavenPack actual count,RavenPack actual %,RavenPack predicted count,RavenPack predicted %
label,,,,,,
negative,604,12.5,19262,28.0,3004,16.5
neutral,2879,59.4,9559,13.9,1353,7.5
positive,1363,28.1,39901,58.1,13797,76.0


## Model provenance & reproducibility snapshot

To be able to reproduce or audit the exact checkpoint evaluated above
(`ckpt_label`, `model_dir`), capture:

1. **Revision hash** — git commit of this codebase at run time (+ dirty flag).
2. **Weights snapshot** — sha256 checksum of each saved weight file (detects any
   accidental re-save, corruption, or silent overwrite).
3. **Config info** — `model.config.to_dict()`, `num_labels`, `id2label`.
4. **Tokenizer settings** — `max_length` and padding strategy used at train time.
5. **Data provenance** — dataset, split sizes, training seed, and a content
   hash per split (detects silent upstream changes to the training data).

Branches on whether `model_dir` is the PhraseBank checkpoint or the RavenPack
fine-tuned checkpoint, since their training data differs. Everything is written
to `provenance.json` next to `metrics.json` in `model_dir`.

In [43]:
import subprocess
import hashlib
import json
from datetime import datetime, timezone

from sentiment_ltr.utils import get_git_info


# ── 1. Git info ───────────────────────────────────────────────────────────────

git_info = get_git_info(PROJECT_ROOT)
git_info


# ── 2. Weights snapshot ───────────────────────────────────────────────────────

def _hash_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """sha256 of a file's bytes — cheap integrity check for saved weights."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


weight_files = sorted(
    p for p in model_dir.glob("*") if p.suffix in {".safetensors", ".bin"}
)
weights_snapshot = [
    {"file": p.name, "size_bytes": p.stat().st_size, "sha256": _hash_file(p)}
    for p in weight_files
]
pd.DataFrame(weights_snapshot)


# ── 3. Model config ───────────────────────────────────────────────────────────

config_dict = model.config.to_dict()
config_info = {
    "num_labels": model.config.num_labels,
    "id2label": model.config.id2label,
    "label2id": model.config.label2id,
    "architectures": config_dict.get("architectures"),
    "model_type": config_dict.get("model_type"),
    "full_config": config_dict,
}
print("num_labels:", config_info["num_labels"])
print("id2label  :", config_info["id2label"])
print("label2id  :", config_info["label2id"])


# ── 4. Tokenizer settings ─────────────────────────────────────────────────────

ckpt_metrics = load_metrics(model_dir)
tokenizer_settings = {
    "tokenizer_class": type(tokenizer).__name__,
    "max_length_used": ckpt_metrics.get("max_length", MAX_LENGTH),
    "padding_strategy": "max_length",  # training-time padding (tokenize_dataset helpers)
    "truncation": True,
    "model_max_length": tokenizer.model_max_length,
    "vocab_size": tokenizer.vocab_size,
}
tokenizer_settings


# ── 5. Data provenance ───────────────────────────────────────────────────────

def _hash_pairs(texts, labels) -> str:
    """Stable content hash over (text, label) pairs — detects silent upstream
    changes to the training data between runs."""
    h = hashlib.sha256()
    for t, l in zip(texts, labels):
        h.update(str(t).encode("utf-8"))
        h.update(b"\x00")
        h.update(str(l).encode("utf-8"))
        h.update(b"\x01")
    return h.hexdigest()


if model_dir == DEFAULT_RAVENPACK_MODEL_DIR:
    # RavenPack fine-tuned checkpoint — trained on the RavenPack labeled frame.
    rp_splits = labeled.copy()
    rp_splits["split"] = assign_time_split(rp_splits["article_date"])
    data_provenance = {
        "dataset_repo": f"RavenPack via WRDS ({TICKER})",
        "dataset_config": SPLIT_SOURCE,
        "split_sizes": rp_splits["split"].value_counts().to_dict(),
        "split_type": "time-based (train<=2011, validation=2012, test>=2013); no random seed",
        "training_seed": 42,  # ravenpack_sentiment.train_ravenpack default
        "split_content_sha256": {
            split_name: _hash_pairs(sub["headline"], sub["label_name"])
            for split_name, sub in rp_splits.groupby("split")
        },
    }
else:
    # PhraseBank checkpoint — trained on Financial PhraseBank (liquidAI_prep.ipynb).
    data_provenance = {
        "dataset_repo": "atrost/financial_phrasebank",
        "dataset_config": "sentences_50agree (via atrost/financial_phrasebank mirror)",
        "split_sizes": {name: raw[name].num_rows for name in raw},
        "split_type": "pre-defined splits shipped by source repo; no re-split seed",
        "training_seed": 42,  # liquidAI_prep.ipynb TrainingArguments(seed=...)
        "split_content_sha256": {
            name: _hash_pairs(raw[name]["sentence"], raw[name]["label"]) for name in raw
        },
    }
data_provenance


# ── 6. Provenance save ───────────────────────────────────────────────────────

provenance = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "checkpoint": {"label": ckpt_label, "path": str(model_dir.relative_to(PROJECT_ROOT))},
    "git": git_info,
    "weights": weights_snapshot,
    "model_config": config_info,
    "tokenizer": tokenizer_settings,
    "data": data_provenance,
}

provenance_path = model_dir / "provenance.json"
provenance_path.write_text(json.dumps(provenance, indent=2, default=str) + "\n", encoding="utf-8")
print(f"Saved provenance snapshot to {provenance_path.resolve()}")
print(f"  checkpoint      : {ckpt_label} ({model_dir.relative_to(PROJECT_ROOT)})")
print(f"  git commit      : {git_info['commit_hash_short']} (dirty={git_info['is_dirty']})")
print(f"  weight files    : {[w['file'] for w in weights_snapshot]}")
print(f"  num_labels      : {config_info['num_labels']}")
print(f"  max_length      : {tokenizer_settings['max_length_used']} (padding={tokenizer_settings['padding_strategy']})")
print(f"  split sizes     : {data_provenance['split_sizes']}")

num_labels: 3
id2label  : {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id  : {'negative': 0, 'neutral': 1, 'positive': 2}
Saved provenance snapshot to /Users/armandoordoricadelatorre/Documents/U of T/PhD/PhD Research/Sentiment_learn_to_rank_paper/data/models/ravenpack_distilbert_best/provenance.json
  checkpoint      : RavenPack fine-tuned (data/models/ravenpack_distilbert_best)
  git commit      : 9e7e106 (dirty=True)
  weight files    : ['model.safetensors', 'training_args.bin']
  num_labels      : 3
  max_length      : 128 (padding=max_length)
  split sizes     : {'train': 37832, 'test': 18154, 'validation': 12736}


In [44]:
# ── 2. Weight-file SHA-256 snapshot ──────────────────────────────────────────
from sentiment_ltr.provenance import get_weight_files_snapshot

weights_snapshot = get_weight_files_snapshot(model_dir)
pd.DataFrame(weights_snapshot)


,file,size_bytes,sha256
0,model.safetensors,267835644,4e50dc5fd511eee090f069fe8a76246fd63646cca149e2...
1,training_args.bin,5393,bc3d33e662444a83eb3feb22d1f30838352377a401c351...


In [45]:
# ── 3. Model config info ──────────────────────────────────────────────────────
config_info = get_model_config_info(model)
print("num_labels:", config_info["num_labels"])
print("id2label  :", config_info["id2label"])
print("label2id  :", config_info["label2id"])


num_labels: 3
id2label  : {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id  : {'negative': 0, 'neutral': 1, 'positive': 2}
 3
id2label  : {0: 'negative', 1: 'neutral', 2: 'positive'}
label2id  : {'negative': 0, 'neutral': 1, 'positive': 2}


In [46]:
# ── 4. Tokenizer settings ─────────────────────────────────────────────────────
ckpt_metrics = load_metrics(model_dir)
tokenizer_info = get_tokenizer_provenance(tokenizer, ckpt_metrics)
tokenizer_info


{'tokenizer_class': 'BertTokenizer',
 'max_length_used': 128,
 'padding_strategy': 'max_length',
 'truncation': True,
 'model_max_length': 512,
 'vocab_size': 30522}

In [47]:
# ── 5. Data provenance ────────────────────────────────────────────────────────
if model_dir == DEFAULT_RAVENPACK_MODEL_DIR:
    data_info = ravenpack_data_provenance(labeled, TICKER)
else:
    data_info = phrasebank_data_provenance(raw)

data_info


{'dataset_repo': 'RavenPack via WRDS (AAPL)',
 'dataset_config': 'RavenPack time split: train ≤2011, val 2012, test ≥2013 (|score|>0.05 → label; else neutral)',
 'split_sizes': {'train': 37832, 'test': 18154, 'validation': 12736},
 'split_type': 'time-based (train<=2011, validation=2012, test>=2013); no random seed',
 'training_seed': 42,
 'split_content_sha256': {'test': '6deb762dde891a66cd3fd44b29c0a1c11ddf5fb3447d4d6840606b2b53429eec',
  'train': 'faaaac48c59f34f3eec602dc1ca39653200b406b19cdd1c320fd693c78e9bfb6',
  'validation': '08fc29e7beac604784026afbaf557a59bc89e51734bf030ebc6a48b17ab51be3'}}

In [48]:
# ── 6. Assemble & save provenance.json ───────────────────────────────────────
provenance = build_checkpoint_provenance(
    checkpoint_label=ckpt_label,
    model_dir=model_dir,
    project_root=PROJECT_ROOT,
    config_info=config_info,
    tokenizer_info=tokenizer_info,
    data_info=data_info,
)

provenance_path = save_provenance(provenance, model_dir)
print(f"Saved provenance snapshot to {provenance_path}")
print_provenance_summary(provenance, PROJECT_ROOT)


Saved provenance snapshot to /Users/armandoordoricadelatorre/Documents/U of T/PhD/PhD Research/Sentiment_learn_to_rank_paper/data/models/ravenpack_distilbert_best/provenance.json
  checkpoint      : RavenPack fine-tuned (data/models/ravenpack_distilbert_best)
  git commit      : 9e7e106 (dirty=True)
  weight files    : ['model.safetensors', 'training_args.bin']
  num_labels      : 3
  max_length      : 128 (padding=max_length)
  split sizes     : {'train': 37832, 'test': 18154, 'validation': 12736}


## Macro-F1 comparison: in-domain PhraseBank vs out-of-domain RavenPack

Compare the same loaded checkpoint across the PhraseBank train / validation / test splits and the RavenPack evaluation split. The first chart shows overall macro-F1 and accuracy; the second chart breaks macro-F1 down by sentiment class so the domain-shift failure mode is easier to see.

In [49]:
_CLASS_ORDER = ["negative", "neutral", "positive"]


def _metrics_row(dataset: str, split: str, domain: str, y_true, y_pred) -> dict:
    return {
        "dataset": dataset,
        "split": split,
        "domain": domain,
        "evaluation": f"{dataset} {split}",
        "checkpoint": ckpt_label,
        "n_rows": int(len(y_true)),
        "macro_f1": float(
            f1_score(y_true, y_pred, labels=_CLASS_ORDER, average="macro", zero_division=0)
        ),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }


def _class_f1_rows(dataset: str, split: str, domain: str, y_true, y_pred) -> list[dict]:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=_CLASS_ORDER, zero_division=0,
    )
    return [
        {
            "dataset": dataset,
            "split": split,
            "domain": domain,
            "evaluation": f"{dataset} {split}",
            "label": label,
            "precision": float(p),
            "recall": float(r),
            "f1": float(f),
            "support": int(s),
        }
        for label, p, r, f, s in zip(_CLASS_ORDER, precision, recall, f1, support)
    ]


summary_rows: list[dict] = []
class_rows: list[dict] = []

for split_name in ["train", "validation", "test"]:
    split_df = raw[split_name].to_pandas()
    sentences = split_df["sentence"].tolist()
    split_preds = pd.concat(
        [predict_sentences(sentences[i : i + 64], tokenizer, model, device)
         for i in range(0, len(sentences), 64)],
        ignore_index=True,
    )
    y_true_split = split_df["label"].map(dataset_id2label)
    y_pred_split = split_preds["pred"]
    summary_rows.append(_metrics_row("PhraseBank", split_name, "in-domain", y_true_split, y_pred_split))
    class_rows.extend(_class_f1_rows("PhraseBank", split_name, "in-domain", y_true_split, y_pred_split))

rp_split = EVAL_SPLIT or "all"
summary_rows.append(_metrics_row("RavenPack", rp_split, "out-of-domain", results["actual"], results["pred"]))
class_rows.extend(_class_f1_rows("RavenPack", rp_split, "out-of-domain", results["actual"], results["pred"]))

f1_comparison = pd.DataFrame(summary_rows)
class_f1_comparison = pd.DataFrame(class_rows)

display(
    f1_comparison[["dataset", "split", "domain", "n_rows", "macro_f1", "accuracy"]]
    .style.format({"macro_f1": "{:.1%}", "accuracy": "{:.1%}", "n_rows": "{:,}"})
)

metric_long = (
    f1_comparison
    .melt(
        id_vars=["dataset", "split", "domain", "evaluation", "checkpoint", "n_rows"],
        value_vars=["macro_f1", "accuracy"],
        var_name="metric",
        value_name="score",
    )
    .assign(
        metric=lambda d: d["metric"].map({"macro_f1": "Macro-F1", "accuracy": "Accuracy"}),
        score_label=lambda d: d["score"].map(lambda x: f"{x:.1%}"),
    )
)

overall_metric_fig = px.bar(
    metric_long,
    x="evaluation", y="score", color="metric", barmode="group", text="score_label",
    hover_data={"domain": True, "checkpoint": True, "n_rows": ":,", "score": ":.3f"},
    title=f"Baseline checkpoint performance: PhraseBank splits vs RavenPack ({rp_split})",
    color_discrete_map={"Macro-F1": "#2563eb", "Accuracy": "#059669"},
)
overall_metric_fig.update_traces(textposition="outside", cliponaxis=False)
overall_metric_fig.update_yaxes(title="Score", tickformat=".0%", range=[0, 1.08])
overall_metric_fig.update_xaxes(title="Evaluation split")
overall_metric_fig.update_layout(
    hovermode="closest",
    height=480, legend_title_text="Metric", margin=dict(t=70, r=30, b=70, l=60),
)
overall_metric_fig.show()

class_f1_comparison["f1_label"] = class_f1_comparison["f1"].map(lambda x: f"{x:.1%}")
class_f1_fig = px.bar(
    class_f1_comparison,
    x="label", y="f1", color="evaluation", barmode="group", text="f1_label",
    facet_col="domain",
    hover_data={"support": ":,", "precision": ":.3f", "recall": ":.3f", "f1": ":.3f"},
    title="Class-level F1 by domain and split",
)
class_f1_fig.update_traces(textposition="outside", cliponaxis=False)
class_f1_fig.update_yaxes(title="Class F1", tickformat=".0%", range=[0, 1.08])
class_f1_fig.update_xaxes(title="Sentiment class")
class_f1_fig.update_layout(
    hovermode="closest",
    height=500, legend_title_text="Evaluation", margin=dict(t=80, r=30, b=60, l=60),
)
class_f1_fig.show()

precision_recall_long = (
    class_f1_comparison
    .melt(
        id_vars=["dataset", "split", "domain", "evaluation", "label", "support"],
        value_vars=["precision", "recall"],
        var_name="metric",
        value_name="score",
    )
    .assign(
        metric=lambda d: d["metric"].str.title(),
        score_label=lambda d: d["score"].map(lambda x: f"{x:.1%}"),
    )
)

precision_recall_fig = px.bar(
    precision_recall_long,
    x="label", y="score", color="metric", barmode="group", text="score_label",
    facet_row="domain", facet_col="evaluation",
    hover_data={"support": ":,", "score": ":.3f"},
    title="Precision vs recall by class, domain, and split",
    color_discrete_map={"Precision": "#7c3aed", "Recall": "#ea580c"},
)
precision_recall_fig.update_traces(textposition="outside", cliponaxis=False)
precision_recall_fig.update_yaxes(title="Score", tickformat=".0%", range=[0, 1.08])
precision_recall_fig.update_xaxes(title="Sentiment class")
precision_recall_fig.update_layout(
    hovermode="closest",
    height=720, legend_title_text="Metric", margin=dict(t=90, r=30, b=60, l=60),
)
precision_recall_fig.for_each_annotation(lambda a: a.update(text=a.text.replace("domain=", "").replace("evaluation=", "")))
precision_recall_fig.show()

precision_recall_drivers = pd.DataFrame([
    {
        "evaluation": row.evaluation,
        "domain": row.domain,
        "label": row.label,
        "precision": row.precision,
        "recall": row.recall,
        "f1": row.f1,
        "lower_component": "recall" if row.recall < row.precision else "precision",
        "precision_minus_recall": row.precision - row.recall,
    }
    for row in class_f1_comparison.itertuples(index=False)
])
display(
    precision_recall_drivers.sort_values(["domain", "evaluation", "label"])
    .style.format({
        "precision": "{:.1%}", "recall": "{:.1%}", "f1": "{:.1%}",
        "precision_minus_recall": "{:+.1%}",
    })
)

pb_test_f1 = f1_comparison.loc[
    (f1_comparison["dataset"] == "PhraseBank") & (f1_comparison["split"] == "test"), "macro_f1"
].iloc[0]
rp_f1 = f1_comparison.loc[f1_comparison["dataset"] == "RavenPack", "macro_f1"].iloc[0]
print(
    f"Domain-shift macro-F1 gap (PhraseBank test → RavenPack {rp_split}): "
    f"{pb_test_f1 - rp_f1:.1%} ({pb_test_f1:.1%} → {rp_f1:.1%})"
)


,dataset,split,domain,n_rows,macro_f1,accuracy
0,PhraseBank,train,in-domain,"3,100",60.2%,63.6%
1,PhraseBank,validation,in-domain,776,56.0%,58.6%
2,PhraseBank,test,in-domain,970,52.4%,58.2%
3,RavenPack,test,out-of-domain,"18,154",49.1%,62.4%


,evaluation,domain,label,precision,recall,f1,lower_component,precision_minus_recall
6,PhraseBank test,in-domain,negative,55.0%,26.4%,35.7%,recall,+28.6%
7,PhraseBank test,in-domain,neutral,94.6%,46.9%,62.7%,recall,+47.7%
8,PhraseBank test,in-domain,positive,42.4%,95.4%,58.7%,precision,-53.0%
0,PhraseBank train,in-domain,negative,75.5%,38.7%,51.2%,recall,+36.8%
1,PhraseBank train,in-domain,neutral,95.5%,53.2%,68.4%,recall,+42.2%
2,PhraseBank train,in-domain,positive,44.7%,96.7%,61.2%,precision,-51.9%
3,PhraseBank validation,in-domain,negative,63.3%,39.2%,48.4%,recall,+24.2%
4,PhraseBank validation,in-domain,neutral,91.7%,45.5%,60.8%,recall,+46.2%
5,PhraseBank validation,in-domain,positive,42.5%,95.4%,58.8%,precision,-52.9%
9,RavenPack test,out-of-domain,negative,58.2%,35.5%,44.1%,recall,+22.7%


Domain-shift macro-F1 gap (PhraseBank test → RavenPack test): 3.2% (52.4% → 49.1%)


## Out-of-domain label prevalence: observed vs predicted

Compare the RavenPack actual label distribution against the checkpoint's predicted label distribution on the same out-of-domain evaluation rows. This makes the distribution shift visible as a prevalence gap, not just as precision / recall / F1.

In [50]:
_CLASS_ORDER = ["negative", "neutral", "positive"]
rp_split = EVAL_SPLIT or "all"
n_eval = len(results)

observed_counts = results["actual"].value_counts().reindex(_CLASS_ORDER, fill_value=0)
predicted_counts = results["pred"].value_counts().reindex(_CLASS_ORDER, fill_value=0)

prevalence = pd.concat(
    [
        pd.DataFrame({"label": observed_counts.index, "count": observed_counts.values, "series": "Observed / actual"}),
        pd.DataFrame({"label": predicted_counts.index, "count": predicted_counts.values, "series": "Predicted by checkpoint"}),
    ],
    ignore_index=True,
).assign(
    pct=lambda d: d["count"] / n_eval,
    pct_label=lambda d: d.apply(lambda r: f"{r['pct']:.1%}<br>n={int(r['count']):,}", axis=1),
    label=lambda d: pd.Categorical(d["label"], categories=_CLASS_ORDER, ordered=True),
)

display(
    prevalence.pivot(index="label", columns="series", values=["count", "pct"])
    .reindex(_CLASS_ORDER)
    .style.format({
        ("pct", "Observed / actual"): "{:.1%}",
        ("pct", "Predicted by checkpoint"): "{:.1%}",
    })
)

ood_prevalence_fig = px.bar(
    prevalence,
    x="label", y="pct", color="series", barmode="group", text="pct_label",
    category_orders={"label": _CLASS_ORDER, "series": ["Observed / actual", "Predicted by checkpoint"]},
    hover_data={"count": ":,", "pct": ":.2%", "label": False},
    title=f"RavenPack label prevalence: observed vs predicted ({ckpt_label}, split={rp_split}, n={n_eval:,})",
    color_discrete_map={"Observed / actual": "#0f766e", "Predicted by checkpoint": "#dc2626"},
)
ood_prevalence_fig.update_traces(textposition="outside", cliponaxis=False)
ood_prevalence_fig.update_yaxes(
    title="Share of out-of-domain rows",
    tickformat=".0%",
    range=[0, max(0.05, prevalence["pct"].max() * 1.18)],
)
ood_prevalence_fig.update_xaxes(title="Sentiment label")
ood_prevalence_fig.update_layout(
    hovermode="closest",
    height=480, legend_title_text="Distribution", margin=dict(t=80, r=30, b=60, l=60),
)
ood_prevalence_fig.show()

gap = prevalence.pivot(index="label", columns="series", values="pct").reindex(_CLASS_ORDER)
gap["prediction_minus_actual_pp"] = (gap["Predicted by checkpoint"] - gap["Observed / actual"]) * 100
print("Prediction prevalence minus actual prevalence (percentage points):")
display(gap[["prediction_minus_actual_pp"]].style.format("{:+.1f} pp"))


Prediction prevalence minus actual prevalence (percentage points):


series,prediction_minus_actual_pp
label,
negative,-10.6 pp
neutral,-7.6 pp
positive,+18.2 pp


## Log RavenPack diagnostics to W&B

Send the same dashboard data to Weights & Biases: scalar summary metrics, W&B Tables, and Plotly figures. Re-run this cell after changing the checkpoint, ticker, split, or evaluation rows so W&B stays in sync with the notebook dashboard.

In [51]:
url = log_ravenpack_diagnostics(
    ticker=TICKER,
    eval_split=EVAL_SPLIT,
    ckpt_label=ckpt_label,
    model_dir=model_dir,
    project_root=PROJECT_ROOT,
    f1_comparison=f1_comparison,
    class_f1_comparison=class_f1_comparison,
    precision_recall_long=precision_recall_long,
    prevalence=prevalence,
    gap=gap,
    overall_metric_fig=overall_metric_fig,
    class_f1_fig=class_f1_fig,
    precision_recall_fig=precision_recall_fig,
    ood_prevalence_fig=ood_prevalence_fig,
)
print(f"Logged RavenPack diagnostics to W&B: {url}")


class_metrics/phrasebank-test/negative/f1,▁
class_metrics/phrasebank-test/negative/precision,▁
class_metrics/phrasebank-test/negative/recall,▁
class_metrics/phrasebank-test/negative/support,▁
class_metrics/phrasebank-test/neutral/f1,▁
class_metrics/phrasebank-test/neutral/precision,▁
class_metrics/phrasebank-test/neutral/recall,▁
class_metrics/phrasebank-test/neutral/support,▁
class_metrics/phrasebank-test/positive/f1,▁
class_metrics/phrasebank-test/positive/precision,▁
+47,...


Logged RavenPack diagnostics to W&B: https://wandb.ai/armando-ordorica-university-of-toronto/sentiment-ltr-transformers/runs/v4zg6j5d


## Checkpoint comparison: PhraseBank baseline vs RavenPack fine-tuned

Side-by-side evaluation of **both** checkpoints on the **same RavenPack test rows** (2013–2014 AAPL, n ≈ 18k).

Re-scores the test split with the PhraseBank checkpoint so every number comes from an identical evaluation set — no cherry-picking, no train/test leakage.


In [ ]:
# ── Re-score same test rows with PhraseBank checkpoint ───────────────────────
_CLASS_ORDER = ["negative", "neutral", "positive"]

# Colour helper (🟥 negative · ⬜ neutral · 🟩 positive)
_SENTIMENT_BG = {
    "negative": "background-color: #fecaca; color: #7f1d1d",
    "neutral":  "background-color: #e2e8f0; color: #1e293b",
    "positive": "background-color: #bbf7d0; color: #14532d",
}
def _color_sentiment(val):
    return _SENTIMENT_BG.get(str(val), "")

pb_tokenizer, pb_model, pb_device = load_classifier(DEFAULT_MODEL_DIR)

pb_pred_chunks: list[pd.DataFrame] = []
_headlines = eval_df["headline"].tolist()
for _start in range(0, len(_headlines), BATCH_SIZE):
    pb_pred_chunks.append(
        predict_sentences(_headlines[_start : _start + BATCH_SIZE], pb_tokenizer, pb_model, pb_device)
    )
pb_preds = pd.concat(pb_pred_chunks, ignore_index=True)

pb_results = (
    eval_df[["split", "article_date", "headline", "event_sentiment_score", "label_name"]]
    .join(pb_preds.drop(columns=["sentence"]))
    .rename(columns={"label_name": "actual"})
)

# ── Build comparison frame ────────────────────────────────────────────────────
cmp_rows = []
for ckpt_name, y_pred_series in [
    ("PhraseBank (no fine-tune)", pb_results["pred"]),
    ("RavenPack fine-tuned",      results["pred"]),
]:
    y_true_series = pb_results["actual"]   # same ground truth for both
    p, r, f, s = precision_recall_fscore_support(
        y_true_series, y_pred_series, labels=_CLASS_ORDER, zero_division=0
    )
    for label, pi, ri, fi, si in zip(_CLASS_ORDER, p, r, f, s):
        cmp_rows.append({
            "checkpoint": ckpt_name,
            "label":      label,
            "precision":  float(pi),
            "recall":     float(ri),
            "f1":         float(fi),
            "support":    int(si),
        })
    cmp_rows.append({
        "checkpoint": ckpt_name,
        "label":      "macro avg",
        "precision":  float(p.mean()),
        "recall":     float(r.mean()),
        "f1":         float(f1_score(y_true_series, y_pred_series, labels=_CLASS_ORDER, average="macro", zero_division=0)),
        "support":    int(len(y_true_series)),
    })

ckpt_cmp = pd.DataFrame(cmp_rows)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"=== RavenPack test set ({TICKER}, {EVAL_SPLIT}, n={len(eval_df):,}) ===\n")
display(
    ckpt_cmp.pivot_table(index="label", columns="checkpoint", values=["precision", "recall", "f1"])
    .reindex(["negative", "neutral", "positive", "macro avg"])
    .round(3)
    .style.format("{:.1%}")
    .bar(align="mid", color=["red", "lightgreen"])
    .set_caption("PhraseBank baseline vs RavenPack fine-tuned — same test rows")
)

# ── Chart 1: Macro-F1 & accuracy side-by-side ─────────────────────────────────
overall_cmp = pd.DataFrame([
    {
        "checkpoint": name,
        "macro_f1":   float(f1_score(pb_results["actual"], preds_s, labels=_CLASS_ORDER, average="macro", zero_division=0)),
        "accuracy":   float(accuracy_score(pb_results["actual"], preds_s)),
    }
    for name, preds_s in [
        ("PhraseBank\n(no fine-tune)", pb_results["pred"]),
        ("RavenPack\nfine-tuned",      results["pred"]),
    ]
])

overall_long = overall_cmp.melt(
    id_vars="checkpoint", value_vars=["macro_f1", "accuracy"],
    var_name="metric", value_name="score",
).assign(
    metric=lambda d: d["metric"].map({"macro_f1": "Macro-F1", "accuracy": "Accuracy"}),
    score_label=lambda d: d["score"].map(lambda x: f"{x:.1%}"),
)

fig_overall = px.bar(
    overall_long,
    x="checkpoint", y="score", color="metric", barmode="group", text="score_label",
    title=f"Overall performance: PhraseBank baseline vs RavenPack fine-tuned<br>"
          f"<sup>RavenPack test set — {TICKER} 2013–2014, n={len(eval_df):,}</sup>",
    color_discrete_map={"Macro-F1": "#2563eb", "Accuracy": "#059669"},
)
fig_overall.update_traces(textposition="outside", cliponaxis=False)
fig_overall.update_yaxes(title="Score", tickformat=".0%", range=[0, 1.08])
fig_overall.update_xaxes(title="Checkpoint")
fig_overall.update_layout(
    hovermode="closest",
    height=480, legend_title_text="Metric", margin=dict(t=90, r=30, b=60, l=60),
)
fig_overall.show()

# ── Chart 2: Per-class F1 ─────────────────────────────────────────────────────
class_cmp = ckpt_cmp[ckpt_cmp["label"] != "macro avg"].copy()
class_cmp["f1_label"] = class_cmp["f1"].map(lambda x: f"{x:.1%}")

fig_class = px.bar(
    class_cmp,
    x="label", y="f1", color="checkpoint", barmode="group", text="f1_label",
    category_orders={"label": _CLASS_ORDER},
    title=f"Per-class F1: PhraseBank baseline vs RavenPack fine-tuned<br>"
          f"<sup>RavenPack test set — {TICKER} 2013–2014, n={len(eval_df):,}</sup>",
    color_discrete_map={
        "PhraseBank (no fine-tune)": "#94a3b8",
        "RavenPack fine-tuned":      "#2563eb",
    },
)
fig_class.update_traces(textposition="outside", cliponaxis=False)
fig_class.update_yaxes(title="Class F1", tickformat=".0%", range=[0, 1.08])
fig_class.update_xaxes(title="Sentiment class")
fig_class.update_layout(
    hovermode="closest",
    height=450, legend_title_text="Checkpoint", margin=dict(t=90, r=30, b=60, l=60),
)
fig_class.show()

# ── Gain summary ─────────────────────────────────────────────────────────────
pb_mf1  = float(f1_score(pb_results["actual"], pb_results["pred"],  labels=_CLASS_ORDER, average="macro", zero_division=0))
rp_mf1  = float(f1_score(pb_results["actual"], results["pred"],     labels=_CLASS_ORDER, average="macro", zero_division=0))
pb_acc  = float(accuracy_score(pb_results["actual"], pb_results["pred"]))
rp_acc  = float(accuracy_score(pb_results["actual"], results["pred"]))
print(f"Macro-F1 : {pb_mf1:.1%} → {rp_mf1:.1%}  ({rp_mf1 - pb_mf1:+.1%} pp)")
print(f"Accuracy : {pb_acc:.1%} → {rp_acc:.1%}  ({rp_acc - pb_acc:+.1%} pp)")

# ── Colour-coded sample: where the two checkpoints disagree ──────────────────
# Build side-by-side frame: PB pred + RP pred + ground truth + probabilities
_prob_cols = ["p(negative)", "p(neutral)", "p(positive)"]
_pb_preds_frame = pb_preds.rename(columns={c: f"pb_{c}" for c in ["pred"] + _prob_cols})
_rp_preds_frame = preds.rename(columns={c: f"rp_{c}" for c in ["pred"] + _prob_cols})

side_by_side = (
    eval_df[["article_date", "headline", "event_sentiment_score", "label_name"]]
    .rename(columns={"label_name": "actual"})
    .join(_pb_preds_frame[["pb_pred", "pb_p(negative)", "pb_p(neutral)", "pb_p(positive)"]])
    .join(_rp_preds_frame[["rp_pred", "rp_p(negative)", "rp_p(neutral)", "rp_p(positive)"]])
    .assign(checkpoints_agree=lambda d: d["pb_pred"] == d["rp_pred"])
)

# Keep only rows where the two models disagree
disagreements = side_by_side[~side_by_side["checkpoints_agree"]].copy()
_dis_sample = disagreements.sample(min(20, len(disagreements)), random_state=42)

_pb_prob_cols = ["pb_p(negative)", "pb_p(neutral)", "pb_p(positive)"]
_rp_prob_cols = ["rp_p(negative)", "rp_p(neutral)", "rp_p(positive)"]
_sentiment_label_cols = ["actual", "pb_pred", "rp_pred"]

print(f"\nHeadlines where checkpoints disagree — sample of {len(_dis_sample)} "
      f"(🟥 negative · ⬜ neutral · 🟩 positive):")
display(
    _dis_sample[["article_date", "headline", "event_sentiment_score",
                 "actual", "pb_pred"] + _pb_prob_cols +
                ["rp_pred"] + _rp_prob_cols]
    .rename(columns={
        "pb_pred": "PhraseBank pred",
        "rp_pred": "RavenPack ft pred",
        "pb_p(negative)": "PB p(neg)", "pb_p(neutral)": "PB p(neu)", "pb_p(positive)": "PB p(pos)",
        "rp_p(negative)": "RP p(neg)", "rp_p(neutral)": "RP p(neu)", "rp_p(positive)": "RP p(pos)",
    })
    .style
    .applymap(_color_sentiment, subset=["actual", "PhraseBank pred", "RavenPack ft pred"])
    .format("{:.3f}", subset=["PB p(neg)", "PB p(neu)", "PB p(pos)",
                               "RP p(neg)", "RP p(neu)", "RP p(pos)"])
    .bar(subset=["PB p(neg)", "PB p(neu)", "PB p(pos)"], align="mid", color=["red", "lightgreen"])
    .bar(subset=["RP p(neg)", "RP p(neu)", "RP p(pos)"], align="mid", color=["red", "lightgreen"])
)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

=== RavenPack test set (AAPL, test, n=18,154) ===



Macro-F1 : 27.5% → 49.1%  (+21.6% pp)
Accuracy : 27.3% → 62.4%  (+35.1% pp)
